# Rede Neural (MLP) para a função lógica XOR com PyTorch

**Disciplina:** Teoria e Aplicações de Inteligência Computacional I — Unidade 1, Aula 3

**Objetivo:** Implementar, em PyTorch, uma Rede Neural Artificial (MLP) capaz de aprender a
função lógica XOR por meio do processo de treinamento.

O XOR é o exemplo clássico de problema não linearmente separável: um único perceptron
não consegue resolvê-lo, sendo necessária pelo menos uma camada oculta com função de
ativação não linear.

**Conjunto de Dados**

| Entrada 1 | Entrada 2 | Saída |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

A implementação segue a estrutura apresentada na Aula 3, Capítulo 5.

## 1. Importação das Bibliotecas

In [ ]:
import torch                                    # Biblioteca principal do PyTorch (tensores e autograd)
import torch.nn as nn                           # Módulo para construção de redes (camadas, ativações, perdas)
import torch.optim as optim                     # Algoritmos de otimização (SGD, Adam) para atualização dos pesos
from torch.utils.data import TensorDataset, DataLoader  # Utilitários para organizar dados em batches
import matplotlib.pyplot as plt                 # Visualização do gráfico da Loss


## 2. Hiperparâmetros

Valores definidos para o problema, conforme a Aula 3.

| Hiperparâmetro           | Valor |
|--------------------------|---|
| Learning Rate ($\eta$)   | 0.001 |
| Número de épocas         | 10000 (com Early Stopping) |
| Batch Size               | 1 (Gradiente Descendente Estocástico) |
| Otimizador               | SGD (momentum = 0.8, weight_decay = 1e-4) |
| Função de perda          | MSELoss (Erro Quadrático Médio) |
| Ativação (camada oculta) | ReLU |
| Ativação (saída)         | Sigmoid |

In [ ]:
LEARNING_RATE = 0.001     # Taxa de aprendizado (eta): tamanho do passo na descida do gradiente
EPOCHS        = 10000     # Número máximo de épocas de treinamento
BATCH_SIZE    = 1         # 1 = SGD | intermediário = Mini-Batch | total (4) = Batch GD
HIDDEN_LAYERS = 1         # Número de camadas ocultas
NEURONS       = 3         # Neurônios por camada oculta
ACTIVATION    = "relu"    # relu | sigmoid | tanh  (ativação da(s) camada(s) oculta(s))
OPTIMIZER     = "SGD"     # SGD | Adam
MOMENTUM      = 0.8        # Termo de momento: acumula gradientes passados para acelerar a convergência
WEIGHT_DECAY  = 0.0001     # Regularização L2 (penaliza pesos grandes, evita overfitting)
DROPOUT       = 0          # Fração de neurônios desativados por passo (0 = desligado)
EARLY_STOPPING = True      # Interrompe o treino se a Loss parar de melhorar
PATIENCE      = 100        # Nº de épocas sem melhora toleradas antes de parar

## 3. Base de dados XOR

Construímos os tensores de entrada (`X`) e saída (`y`) diretamente da tabela verdade,
usando `torch.tensor`. Usamos `float` porque a rede trabalha com valores contínuos.

In [ ]:
# BASE XOR (provinda da tabela verdade)
X = torch.tensor([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
])

y = torch.tensor([
    [0.],   # 0 XOR 0 = 0
    [1.],   # 0 XOR 1 = 1
    [1.],   # 1 XOR 0 = 1
    [0.]    # 1 XOR 1 = 0
])

print("Entradas (X):\n", X)
print("Saídas  (y):\n", y)

## 4. Dataset e DataLoader

`TensorDataset` agrupa `X` e `y` em pares indexados. O `DataLoader` itera sobre esses pares
em batches (aqui, tamanho 1) e embaralha (`shuffle=True`) a cada época.

In [ ]:
dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Total de amostras: {len(dataset)} | Batches por época: {len(loader)}")

## 5. Escolha da função de ativação


In [ ]:
if ACTIVATION.lower() == "relu":
    activation = nn.ReLU()
elif ACTIVATION.lower() == "sigmoid":
    activation = nn.Sigmoid()
elif ACTIVATION.lower() == "tanh":
    activation = nn.Tanh()
else:
    raise ValueError("Função de ativação inválida.")

print("Ativação da camada oculta:", activation)

## 6. Construção da rede neural (MLP)

In [ ]:
layers = []                                   # lista que armazenará as camadas em sequência

# Primeira camada: 2 entradas -> NEURONS
layers.append(nn.Linear(2, NEURONS))
layers.append(activation)

if DROPOUT > 0:
    layers.append(nn.Dropout(DROPOUT))        # regularização opcional

# Camadas ocultas adicionais (a primeira já foi criada, por isso HIDDEN_LAYERS - 1)
for _ in range(HIDDEN_LAYERS - 1):
    layers.append(nn.Linear(NEURONS, NEURONS))
    layers.append(activation)
    if DROPOUT > 0:
        layers.append(nn.Dropout(DROPOUT))

# Camada de saída: NEURONS -> 1, seguida de Sigmoid
layers.append(nn.Linear(NEURONS, 1))
layers.append(nn.Sigmoid())

# nn.Sequential aplica as camadas uma após a outra no forward pass
model = nn.Sequential(*layers)
print(model)

## 7. Função de perda e otimizador

- **Perda:** `MSELoss` (Erro Quadrático Médio)
- **Otimizador:** `SGD` com `momentum` (acelera a convergência) e `weight_decay` (L2).

In [ ]:
# FUNÇÃO DE PERDA
criterion = nn.MSELoss()

# OTIMIZADOR
if OPTIMIZER.upper() == "SGD":
    optimizer = optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )
elif OPTIMIZER.upper() == "ADAM":
    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
else:
    raise ValueError("Otimizador inválido. Use 'SGD' ou 'Adam'.")

print("Perda:", criterion)
print("Otimizador:", optimizer)